#### 1. Pipeline Design and Medallion

1. **“Design a pipeline that gets daily CSV files from ADLS/S3 into a dashboard.”**

   Auto Loader → Bronze (raw + `_ingest_time`, `_file_name`) → Silver (types, dedupe, nulls, SCD2) → Gold (star schema / aggregates) → Databricks SQL / Power BI. Run it as a job with dependencies, plus alerts on failure.

2. **“Why 3 layers? Why not go straight to Gold?”**

   Bronze lets you **reprocess** without going back to the source. Silver is the **single clean truth** reused by many Gold tables. Gold is shaped for the business.

3. **“What goes in Bronze: raw or cleaned?”**

   Raw, as-is, append-only, with audit columns. No business logic.

4. **“Source sends the same file twice. What happens?”**

   Auto Loader / `COPY INTO` track processed files. In Silver, dedupe with `ROW_NUMBER()` on the key + latest timestamp, and load with **MERGE**, so re-runs are **idempotent**.

5. **“Source added a new column overnight.”**

   Auto Loader schema evolution (`cloudFiles.schemaEvolutionMode = addNewColumns`), `mergeSchema` on write, and `_rescued_data` for unexpected fields.

#### 2. Incremental Loads and CDC

6. **“Table has 2 billion rows. You can’t reload daily. How do you load only changes?”**

   Watermark column (`updated_at > last_max`), or CDC from source, then `MERGE INTO`. Or Delta **Change Data Feed** between layers.

7. **“How do you handle deletes from the source?”**

   CDC flag (`op = 'D'`) → `WHEN MATCHED AND op='D' THEN DELETE`, or a soft-delete flag `is_deleted`.

8. **“Customer changes address. Business wants history.”**

   SCD2 with MERGE (what you just built), or `APPLY CHANGES ... STORED AS SCD TYPE 2` in DLT/Lakeflow.

9. **“Late-arriving data: an order for yesterday arrives today.”**

   MERGE on the key (not append), partition/cluster by date, and pick the dimension version valid on the **order date**.

#### 3. Performance (Very Common)

10. **“A job that took 20 min now takes 2 hours. How do you debug?”**

    Spark UI: find the slow stage → check **data skew** (one task much longer), **spills**, too many small files, a changed data volume, or a missing filter/partition pruning.

11. **“Join between a 1 TB table and a 10 MB table is slow.”**

    **Broadcast join** (`broadcast(small_df)`).

12. **“One key has 80% of the data (skew).”**

    AQE skew join (`spark.sql.adaptive.skewJoin.enabled`), **salting** the key, or handling the hot key separately.

13. **“Thousands of tiny files.”**

    `OPTIMIZE`, auto-compaction/optimized writes, and fewer write partitions (`coalesce`).

14. **“Partitioning vs Z-ORDER vs Liquid Clustering?”**

    Partition only by low-cardinality columns on huge tables (date). Z-ORDER/Liquid Clustering for high-cardinality filter columns (customer_id). New tables: **Liquid Clustering**.

15. **“cache() vs persist()?”**

    Use them when a DataFrame is reused many times. Unpersist after. Not useful for a single pass.

#### 4. Delta Lake

16. **“Someone ran a wrong UPDATE on prod. Fix it.”**

    `DESCRIBE HISTORY` → `RESTORE TABLE t TO VERSION AS OF 12`. That’s time travel.

17. **“Two jobs write to the same table at once.”**

    Delta ACID optimistic concurrency. A conflict fails one writer, and it retries. Avoid by partition-scoped MERGE conditions.

18. **“Storage cost keeps growing.”**

    `VACUUM` old files (default 7 days retention), `OPTIMIZE`, and lifecycle rules on Bronze.

#### 5. Jobs, Orchestration and Failures

19. **“5 notebooks with dependencies. Task 3 failed at 2 AM. What now?”**

    Job alert email/Teams → check task output/logs → fix → **Repair run** (re-runs only failed + downstream tasks). Use retries for flaky sources.

20. **“Pass a value from task 1 to task 2.”**

    `dbutils.jobs.taskValues.set/get` (you’ve done this).

21. **“Run only if the file has arrived.”**

    File-arrival trigger, or a condition task (`if/else`).

22. **“How do you make a job re-runnable safely?”**

    Idempotent writes: MERGE or `replaceWhere` for a date partition, never blind append.

#### 6. Data Quality

23. **“Bad data reached the dashboard. How do you prevent it?”**

    Checks in Silver: not-null, unique keys, ranges. DLT **expectations** (`expect_or_drop`, `expect_or_fail`), and quarantine bad rows to a separate table.

24. **“Row counts don’t match between source and target.”**

    Reconciliation: counts + sums per day between source, Bronze and Silver, logged in an audit table.

#### 7. Governance and Security (Unity Catalog)

25. **“Analysts must not see PII (phone, email).”**

    Column masks / row filters in UC, or a Gold view without PII. Grants on catalog/schema/table to groups, not users.

26. **“Where did this Gold column come from?”**

    Unity Catalog **lineage**.

27. **“How do you store the DB password?”**

    Secret scopes / Key Vault: `dbutils.secrets.get(...)`. Never in code.

#### 8. Streaming

28. **“Near-real-time orders from Kafka/Event Hub.”**

    Structured Streaming → Bronze with **checkpoint**, `trigger(availableNow=True)` for batch-like runs or continuous for low latency, and a watermark for late events.

29. **“Stream restarted. Will data duplicate?”**

    No, if the checkpoint is kept and the sink is Delta (exactly-once). Don’t delete the checkpoint.

#### 9. Your SAS Migration (They WILL Ask This, Be Ready)

30. **“Tell me about your project.”**

    “We migrated SAS ETL to PySpark on Databricks. I converted SAS DATA steps / PROC SQL into PySpark notebooks, built jobs with task dependencies, and loaded Bronze → Silver → Gold Delta tables.”

31. **“How did you validate the PySpark output matches SAS?”**

    Row counts, column sums, and a full `exceptAll` diff between the SAS output and Spark output for sample periods.

32. **“SAS `RETAIN` / `FIRST.` / `LAST.` in PySpark?”**

    Window functions: `lag`, running `sum().over(...)`, `row_number()` for first/last per group.

33. **“Biggest challenge?”**

    Pick a real one: date formats, missing-value differences (SAS `.` vs null), sort-dependent logic, or performance of a big join, and **what you did**.

#### 10. CI/CD (You’ve Done It)

34. **“How do you move code from dev to prod?”**

    Git branch → PR → CI (tests + `bundle validate`) → merge → CD deploys with Asset Bundles using a service principal, with an approval before prod.
